# GenMusic — Full Data Prep: Cleaning + G2P Phonemization + Tokenization (IDs Output)

Notebook này xử lý trọn gói từ A-Z cho tập dữ liệu raw audio trên Kaggle:
1. **Cleaning**: Lọc rác, Mojibake, quảng cáo YouTube (`clean_vietnamese_lyric`).
2. **G2P Phonemize**: Chuyển lyric sạch sang IPA Phoneme bằng ByT5 (`charsiu/g2p_multilingual_byT5_small_100`).
3. **Tokenization (IDs Output)**: Mã hóa chuỗi Phoneme trực tiếp sang mảng số nguyên `phoneme_ids` và `attention_mask` bằng XPhoneBERT (`vinai/xphonebert-base`).

⚡ **Kết quả**: File `records.jsonl` chứa sẵn `phoneme_ids`. Khi huấn luyện mô hình (Training), PyTorch DataLoader truyền thẳng `phoneme_ids` vào model $\rightarrow$ **Tốc độ train tăng tối đa, không tốn 1ms nào cho G2P/Tokenize trong lúc train.**

In [ ]:
# ── Cell 1: Setup & Kiểm tra môi trường ──────────────────────────────────────────────
import os, sys, json, shutil, time, re, unicodedata
from pathlib import Path
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

INPUT_BASE = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/processed_pretokenized_dataset')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'waveforms').mkdir(exist_ok=True)

print(f'OUTPUT_DIR: {OUTPUT_DIR}')

In [ ]:
# ── Cell 2: Module Cleaning Tiếng Việt (lyric_quality.py) ───────────────────
_VIETNAMESE_MARKED_CHARS = frozenset('ăâđêôơưáàảãạấầẩẫậắằẳẵặéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ')
_VIETNAMESE_COMMON_WORDS = frozenset('ai anh ba bao bay ben biet bon buoc ca cac can chang chi cho chung co con cua da dang dau day dem den dieu doi dung duoc em gi giua hay hon khi khong hai la lai lam len long luc ma mai minh mot mua nam nay nghe ngay nguoi nhau nhieu nhung noi o qua ra rang roi sau se ta thay thi theo them tren trong troi tu van ve vi voi yeu'.split())
_ENGLISH_COMMON_WORDS = frozenset('a an and are be been but by can come comes do for from have i in is it let love me my of on open our say scale sign since subscribe taking the this to was we with you your'.split())
_TRANSCRIPT_NOISE_PHRASES = ('dang ky kenh', 'hay subscribe', 'subscribe cho kenh', 'cam on cac ban')
_MOJIBAKE_SEQUENCES = ('Æ°', 'Æ¡', 'Ä‘', 'Äƒ', 'áº', 'á»', 'â€', 'ðŸ')
_UTF8_LATIN1_PREFIX = re.compile(r'Ã[\u00a0-\u00bf]')

def text_has_mojibake(text: str) -> bool:
    val = str(text or '')
    if '\ufffd' in val or any(s in val for s in _MOJIBAKE_SEQUENCES):
        return True
    if _UTF8_LATIN1_PREFIX.search(val):
        return True
    return any('\u0080' <= c <= '\u009f' for c in val)

def _accentless_token(val: str) -> str:
    dec = unicodedata.normalize('NFD', str(val).casefold()).replace('đ', 'd')
    return ''.join(c for c in dec if unicodedata.category(c) != 'Mn')

def clean_vietnamese_lyric(text: str) -> str:
    norm = unicodedata.normalize('NFKC', str(text or '')).strip()
    if not norm or text_has_mojibake(norm):
        return ''
    letters = [c for c in norm if c.isalpha()]
    non_latin = [c for c in letters if 'LATIN' not in unicodedata.name(c, '') and c.casefold() != 'đ']
    if non_latin and len(non_latin) / max(1, len(letters)) > 0.02:
        return ''
    cleaned = re.sub(r'\s+', ' ', ''.join(c if c.isalpha() or c.isspace() or c in "'-.,!?" else ' ' for c in norm)).strip(" -'.,!?")
    words = re.findall(r'[^\W\d_]+', cleaned.casefold(), flags=re.UNICODE)
    if len(words) < 2:
        return ''
    folded_words = [_accentless_token(w) for w in words]
    folded_text = ' '.join(folded_words)
    if any(p in folded_text for p in _TRANSCRIPT_NOISE_PHRASES):
        return ''
    vn_hits = sum(w in _VIETNAMESE_COMMON_WORDS for w in folded_words)
    en_hits = sum(w in _ENGLISH_COMMON_WORDS for w in folded_words)
    marked_cnt = sum(c.casefold() in _VIETNAMESE_MARKED_CHARS for c in cleaned)
    if vn_hits < 2 and marked_cnt < 1:
        return ''
    if en_hits >= 2 and en_hits >= vn_hits:
        return ''
    return cleaned

# Test cleaning
print('Test clean: "Đăng ký kênh cảm ơn các bạn" →', repr(clean_vietnamese_lyric("Đăng ký kênh cảm ơn các bạn")))
print('Test clean: "Một chiều mưa tôi nhớ em" →', repr(clean_vietnamese_lyric("Một chiều mưa tôi nhớ em")))

In [ ]:
# ── Cell 3: Load G2P ByT5 + XPhoneBERT Tokenizer ────────────────────────────
from transformers import T5ForConditionalGeneration, AutoTokenizer

G2P_MODEL_ID = 'charsiu/g2p_multilingual_byT5_small_100'
XPHONEBERT_MODEL_ID = 'vinai/xphonebert-base'
LANGUAGE = 'vie-c'
MAX_LEN = 128

print('Loading G2P Tokenizer & Model...')
g2p_tokenizer = AutoTokenizer.from_pretrained(G2P_MODEL_ID)
g2p_model = T5ForConditionalGeneration.from_pretrained(G2P_MODEL_ID).eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
try:
    g2p_model = g2p_model.to(device)
    # Test GPU generate
    _t = g2p_tokenizer('<vie-c> test', return_tensors='pt').to(device)
    with torch.no_grad():
        g2p_model.generate(**_t, max_length=10)
    print(f'✅ G2P loaded on {device}')
except Exception as e:
    print(f'⚠️ Fallback G2P to CPU ({e})')
    device = torch.device('cpu')
    g2p_model = g2p_model.to(device)

print('Loading XPhoneBERT Tokenizer...')
xphonebert_tokenizer = AutoTokenizer.from_pretrained(XPHONEBERT_MODEL_ID)
print('✅ XPhoneBERT Tokenizer ready')

def g2p_batch(texts: list[str]) -> list[str]:
    if not texts:
        return []
    prefixed = [f'<{LANGUAGE}> {t}' if t.strip() else '' for t in texts]
    non_empty_idx = [i for i, t in enumerate(prefixed) if t]
    non_empty_texts = [prefixed[i] for i in non_empty_idx]
    results = [''] * len(texts)
    if not non_empty_texts:
        return results
    inputs = g2p_tokenizer(non_empty_texts, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out_ids = g2p_model.generate(**inputs, max_length=400)
    decoded = g2p_tokenizer.batch_decode(out_ids, skip_special_tokens=True)
    for i, idx in enumerate(non_empty_idx):
        results[idx] = decoded[i]
    return results

In [ ]:
# ── Cell 4: Tìm input dataset & Link Waveforms ───────────────────────────────
records_in = next(INPUT_BASE.rglob('records.jsonl'), None)
if not records_in:
    raise FileNotFoundError('Không tìm thấy records.jsonl trong /kaggle/input!')
input_dir = records_in.parent
print(f'Input dir: {input_dir}')

config_in = input_dir / 'config.json'
if config_in.exists():
    shutil.copy2(config_in, OUTPUT_DIR / 'config.json')
    print('Copied config.json')

waveforms_in = input_dir / 'waveforms'
if waveforms_in.exists():
    pt_files = list(waveforms_in.glob('*.pt'))
    print(f'Linking {len(pt_files)} .pt files...')
    for pt in pt_files:
        dst = OUTPUT_DIR / 'waveforms' / pt.name
        if not dst.exists():
            try:
                dst.symlink_to(pt.resolve())
            except OSError:
                shutil.copy2(pt, dst)
    print('Link waveforms complete!')

In [ ]:
# ── Cell 5: Pipeline: Clean -> Phonemize -> Tokenize (IDs Output) ────────────
BATCH_SIZE = 8
records_out_path = OUTPUT_DIR / 'records.jsonl'
stats = {'total': 0, 'cleaned': 0, 'rejected': 0}
t_start = time.time()

all_records = []
with open(records_in, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            all_records.append(json.loads(line))

print(f'Loaded {len(all_records)} total records')
print(f'Processing in batches of {BATCH_SIZE}...')

with open(records_out_path, 'w', encoding='utf-8') as fout:
    for start_idx in range(0, len(all_records), BATCH_SIZE):
        batch = all_records[start_idx:start_idx + BATCH_SIZE]
        
        clean_batch_texts = []
        valid_recs_in_batch = []
        
        for rec in batch:
            stats['total'] += 1
            raw_text = str(rec.get('text') or rec.get('lyrics') or '')
            cleaned = clean_vietnamese_lyric(raw_text)
            if not cleaned:
                stats['rejected'] += 1
                continue
            
            rec['cleaned_text'] = cleaned
            clean_batch_texts.append(cleaned)
            valid_recs_in_batch.append(rec)
        
        if clean_batch_texts:
            # Batch G2P
            phonemes_list = g2p_batch(clean_batch_texts)
            
            for rec, phonemes in zip(valid_recs_in_batch, phonemes_list):
                rec['phoneme_text'] = phonemes
                
                # Pre-tokenize to XPhoneBERT IDs
                enc = xphonebert_tokenizer(
                    phonemes,
                    padding=False,
                    truncation=True,
                    max_length=MAX_LEN,
                )
                rec['phoneme_ids'] = enc['input_ids']
                rec['attention_mask'] = enc['attention_mask']
                
                fout.write(json.dumps(rec, ensure_ascii=False) + '\n')
                stats['cleaned'] += 1
        
        elapsed = time.time() - t_start
        print(f'  [{stats["total"]}/{len(all_records)}] cleaned={stats["cleaned"]}, rejected={stats["rejected"]} | {elapsed:.0f}s elapsed')

print(f'\n✅ DONE! Total={stats["total"]}, Cleaned&Tokenized={stats["cleaned"]}, Rejected={stats["rejected"]}')
print(f'Total time: {time.time()-t_start:.1f}s')

In [ ]:
# ── Cell 6: Inspect Output ───────────────────────────────────────────────────
print('=== VERIFICATION OF PRE-TOKENIZED OUTPUT ===')
with open(records_out_path, encoding='utf-8') as f:
    first_line = f.readline()
    sample = json.loads(first_line)

print('Record ID     :', sample.get('id'))
print('Cleaned Text  :', repr(sample.get('cleaned_text')))
print('Phoneme Text  :', repr(sample.get('phoneme_text')))
print('Phoneme IDs   :', sample.get('phoneme_ids')[:15], '... (len:', len(sample.get('phoneme_ids')), ')')
print('Attention Mask:', sample.get('attention_mask')[:15], '... (len:', len(sample.get('attention_mask')), ')')
print('\n✅ All checks passed! Dataset is ready for ultra-fast training.')